# 03. Respuestas, evaluación y procedencia de las decisiones

La campaña final conserva las respuestas aceptadas, los fallos del contrato y las puntuaciones consolidadas. Este cuaderno comprueba cómo encajan esas piezas y permite leer una respuesta junto a su evidencia. No genera respuestas ni solicita nuevos juicios.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# Jupyter puede abrir el cuaderno desde la raíz técnica o desde notebooks/.
HERE = Path.cwd().resolve()
SUPPORT = next(p / 'notebooks' for p in (HERE, *HERE.parents)
               if (p / 'notebooks/notebook_support.py').is_file())
sys.path.insert(0, str(SUPPORT))
from notebook_support import find_root, read_jsonl, new_run, run_script, compare_csv

ROOT = find_root()
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)

In [2]:
CAMPAIGN = ROOT / 'outputs/local_runs/generation_scaling_v3'
golden = pd.DataFrame(read_jsonl(ROOT / 'data/golden/variants/generation_golden_v3.jsonl'))
models = {'gemma4:e4b': 'gemma4_e4b', 'qwen3.5:4b': 'qwen35_4b', 'qwen3.5:9b': 'qwen35_9b'}
conditions = ['bm25', 'dense', 'rrf', 'rerank']
accepted, failed = [], []
# La lista cerrada excluye pruebas previas de otros modelos que siguen archivadas.
for model, slug in models.items():
    for condition in conditions:
        path = CAMPAIGN / f'rag_{condition}_top5_{slug}.jsonl'
        accepted.extend(read_jsonl(path))
        failure_path = path.with_name(path.stem + '_failures.jsonl')
        if failure_path.exists():
            failed.extend(read_jsonl(failure_path))
assert len(accepted) == 604 and len(failed) == 20

## Cobertura de la generación

Una respuesta aceptada cumple el contrato de salida; eso no garantiza que sea correcta. Los fallos finales se cuentan aparte y no reciben una puntuación de calidad inventada.

In [3]:
keys = ['qid', 'generator_model', 'retrieval_condition']
responses = pd.DataFrame(accepted)
failures = pd.DataFrame(failed)
assert not responses.duplicated(keys).any()
assert not failures.duplicated(keys).any()
assert set(map(tuple, responses[keys].values)).isdisjoint(set(map(tuple, failures[keys].values)))
counts = responses.groupby(['generator_model', 'retrieval_condition']).size().rename('aceptadas').to_frame()
counts = counts.join(failures.groupby(['generator_model', 'retrieval_condition']).size().rename('fallos')).fillna(0).astype(int)
counts['previstas'] = 52
assert (counts['aceptadas'] + counts['fallos']).eq(counts['previstas']).all()
display(counts)

aceptadas  fallos  previstas
generator_model retrieval_condition                              
gemma4:e4b      bm25                        48       4         52
                dense                       49       3         52
                rerank                      47       5         52
                rrf                         44       8         52
qwen3.5:4b      bm25                        52       0         52
                dense                       52       0         52
                rerank                      52       0         52
                rrf                         52       0         52
qwen3.5:9b      bm25                        52       0         52
                dense                       52       0         52
                rerank                      52       0         52
                rrf                         52       0         52

In [4]:
assert sum(row['generation_attempts'] for row in accepted) == 614
assert sum(row['generation_attempts'] for row in failed) == 60
display(failures[['qid', 'generator_model', 'retrieval_condition', 'generation_attempts', 'error']])

,qid,generator_model,retrieval_condition,generation_attempts,error
0,q-v1-003__explicit,gemma4:e4b,bm25,3,generation did not follow the three-field text contract
1,q-v3-003__explicit,gemma4:e4b,bm25,3,generation did not follow the three-field text contract
2,q-v3-003__implicit,gemma4:e4b,bm25,3,generation did not follow the three-field text contract
3,q-v3-004__explicit,gemma4:e4b,bm25,3,generation did not follow the three-field text contract
4,q-v1-017__explicit,gemma4:e4b,dense,3,citation outside retrieved contexts
5,q-v3-003__explicit,gemma4:e4b,dense,3,generation did not follow the three-field text contract
6,q-v3-004__explicit,gemma4:e4b,dense,3,generation did not follow the three-field text contract
7,q-v1-003__explicit,gemma4:e4b,rrf,3,generation did not follow the three-field text contract
8,q-v1-003__implicit,gemma4:e4b,rrf,3,generation did not follow the three-field text contract
9,q-v1-013__explicit,gemma4:e4b,rrf,3,citation outside retrieved contexts


## Puntuaciones y aplicabilidad

La evaluación final utiliza un instrumento ordinal propio. Fundamentación, corrección y conducta tienen escala 0–2; el criterio de citas es binario. Estos valores no son las métricas RAGAS de las campañas anteriores.

La conducta esperada se toma de las preguntas, porque algunas filas históricas de evaluación no la contienen. Los valores no aplicables se mantienen como ausentes, sin sustituirlos por cero.

In [5]:
verdicts = pd.DataFrame(read_jsonl(CAMPAIGN / 'eval_final_v3.jsonl'))
assert len(verdicts) == 604 and not verdicts.duplicated(keys).any()
assert set(map(tuple, verdicts[keys].values)) == set(map(tuple, responses[keys].values))
verdicts = verdicts.drop(columns=['expected_behavior']).merge(
    golden[['qid', 'expected_behavior', 'explicitness']], on='qid', validate='many_to_one')
criteria = ['faithfulness', 'factual_correctness', 'behavior_ok', 'citations_ok']
numeric = verdicts[criteria].apply(pd.to_numeric, errors='coerce')
display(numeric.isna().sum().rename('sin_puntuacion_numerica').to_frame())
display(verdicts.groupby('expected_behavior').size().rename('respuestas').to_frame())

,sin_puntuacion_numerica
faithfulness,0
factual_correctness,4
behavior_ok,0
citations_ok,0


,respuestas
expected_behavior,
abstain,95
answer,389
grounded_correction,120


## Consenso y validación humana

La procedencia se conserva por criterio. Una respuesta revisada por el autor puede mantener consenso automático en los criterios que no estaban disputados. El acta también contiene una convención de graduación; esa entrada no es una respuesta adicional.

In [6]:
provenance = pd.DataFrame(verdicts['final_provenance'].tolist())
provenance_counts = pd.DataFrame({criterion: provenance[criterion].value_counts() for criterion in criteria}).fillna(0).astype(int)
assert provenance.notna().all().all()
assert verdicts['human_validated_cell'].sum() == 158
display(provenance_counts)
display(verdicts['human_validated_cell'].value_counts().rename_axis('celda_con_revision_humana').to_frame('celdas'))

,faithfulness,factual_correctness,behavior_ok,citations_ok
consenso_jueces,562,529,525,567
validacion_humana,42,75,79,37


,celdas
celda_con_revision_humana,
False,446
True,158


In [7]:
act = read_jsonl(CAMPAIGN / 'arbitraje_discrepancias_v3/validacion_humana_v3.jsonl')
reviews = [row for row in act if 'qid' in row]
assert len(reviews) == 158
final_by_key = verdicts.set_index(keys)
decisions = []
for row in reviews:
    key = tuple(row[field] for field in keys)
    for criterion, decision in row['decisions'].items():
        assert final_by_key.loc[key, criterion] == decision['final']
        assert final_by_key.loc[key, 'final_provenance'][criterion] == 'validacion_humana'
        decisions.append({'criterio': criterion, 'accion': decision['action']})
assert len(decisions) == 233
display(pd.crosstab(pd.DataFrame(decisions)['criterio'], pd.DataFrame(decisions)['accion'], margins=True))

accion,confirmado,corregido,All
criterio,,,
behavior_ok,71,8,79
citations_ok,32,5,37
factual_correctness,64,11,75
faithfulness,38,4,42
All,205,28,233


## Leer una respuesta y sus fragmentos

El ejemplo fija la celda completa: pregunta, generador y recuperador. Cambiar solo el identificador de pregunta no basta para localizar una respuesta concreta.

In [8]:
selection = (responses['qid'] == 'q-v1-001__explicit') & (responses['generator_model'] == 'gemma4:e4b') & (responses['retrieval_condition'] == 'bm25')
example = responses.loc[selection].iloc[0]
print('Pregunta:', example['question'])
print('\nRespuesta:', example['response'])
print('\nConducta declarada:', example['reported_behavior'])
print('\nReferencia:', example['reference'])
display(verdicts.loc[(verdicts[keys] == example[keys]).all(axis=1), keys + criteria + ['final_provenance']])

Pregunta: How does increasing the sample size affect the sampling distribution of the sample mean?

Respuesta: Increasing the sample size causes the error bound to decrease, resulting in a narrower confidence interval [openstax_intro_stats_2e:8.1:chunk-011].

Conducta declarada: answer

Referencia: As the sample size increases, the sampling distribution of the sample mean becomes more nearly normal, retains the population mean, and has a smaller standard deviation equal to the population standard deviation divided by the square root of the sample size.


,qid,generator_model,retrieval_condition,faithfulness,factual_correctness,behavior_ok,citations_ok,final_provenance
0,q-v1-001__explicit,gemma4:e4b,bm25,2,0,0,1,"{'faithfulness': 'consenso_jueces', 'factual_correctness': 'consenso_jueces', 'behavio..."


In [9]:
assert len(example['retrieved_chunk_ids']) == len(example['retrieved_contexts']) == 5
for chunk_id, context in zip(example['retrieved_chunk_ids'], example['retrieved_contexts']):
    print('\n' + chunk_id + '\n' + context + '\n')


openstax_principles_ds:4.1:chunk-016
Calculate a 95% confidence interval for the true population proportion of adults who smoke. Also, 
determine if there has been a statistically significant change in the proportion of smokers as compared to 
five years ago.

Solution

For a 95% confidence interval, the corresponding critical value is 
.

Start off by calculating the sample proportion :

Verify that the normal approximation to the binomial distribution is appropriate by ensuring that both 
and 
 are both at least 5, where  represents the sample proportion.

For this example, 
, and 
. Both of these 
results are at least 5, which verifies that the normal approximation to the binomial distribution is 
appropriate.

The margin of error is then calculated as follows:

The 95% confidence interval is calculated as follows:

Concluding statement:

The researcher can be 95% confident that the true population of adult smokers in the United States is 
between 0.218 and 0.262, which can also be

Las tablas de este cuaderno describen cobertura y procedencia, no comparan medias de calidad. Para comparar estrategias hay que igualar el soporte disponible y agregar por pregunta base, como hace el análisis del cuaderno 04.